# Lohnt sich das? – Der Kontroll-Rechner
**Team 5 (Roland, Leo, Tobias)** · Bonus

Unser Modell sagt **wer** ist verdächtig. Aber es sagt nicht **wie viele** Kunden STEG soll kontrollieren. Jede Kontrolle kostet Geld und nicht jede Kontrolle findet ein Betrüger. In diesem Notebook wir rechnen deshalb aus, bei wie vielen Kontrollen bleibt am meisten Geld übrig.

**So benutzt ihr den Rechner:** In Abschnitt 2 die zwei Zahlen ändern, dann *Run All*. Alle Ergebnisse und Grafiken weiter unten rechnen sich dann neu aus. Die Texte unter den Grafiken natürlich nicht, die gelten für die Standardwerte.

**Voraussetzung:** 04 bis 07 müssen sein einmal ausgeführt worden. Die Dateien data/fd_features.csv und models/fd_best_model.joblib müssen existieren.

## 1. Setup

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# clone = eine frische, untrainierte Kopie vom Modell machen
# cross_val_predict = für jeden Kunden eine Vorhersage aus dem Fold, in dem er nicht im Training war
from sklearn.base import clone
from sklearn.model_selection import cross_val_predict

# die selben 5 Folds wie in den anderen Notebooks
from fd_utils import cv

sns.set_theme(style="whitegrid")

DATA_DIR = Path("data")
MODEL_DIR = Path("models")

## 2. Eingaben
**Hier ihr dürft spielen.** Die Standardwerte sind unsere Schätzung:

* **Kosten pro Kontrolle: 25 €** – eine Studie aus Europa kommt auf ca. 22,55 € pro Vor-Ort-Kontrolle, mit Personal, Anfahrt und Ausrüstung. In Tunesien sind die Löhne niedriger, dafür muss bei einem Fund noch ein Gerichtsvollzieher kommen.
* **Wert eines gefundenen Betrugs: 150 €** – ca. 100 € Nachzahlung, das ist etwa die Hälfte von 110 kWh pro Monat, 2 Jahre lang, zum tunesischen Strompreis, plus ca. 50 € weil der Betrug danach aufhört.

Wichtig: das sind **Annahmen** von uns, es sind keine echte Zahlen von STEG.

In [ ]:
KOSTEN_PRO_KONTROLLE = 25   # Euro, was kostet eine Kontrolle vor Ort
WERT_PRO_BETRUG = 150       # Euro, was ist ein gefundene Betrug wert für STEG

# Optional: Wie viele Kontrollen die Kontrolleure schaffen maximal? None = keine Grenze
# zum Beispiel 5000 eintragen, wenn es gibt nur Personal für 5000 Kontrollen
MAX_KONTROLLEN = None

### Vorhersagen für jeden Kunden
Für den Rechner brauchen wir für jeden Kunden eine Betrugs-Wahrscheinlichkeit. Die muss kommen von ein Modell, das hat diesen Kunde **nicht** gesehen im Training, sonst wären die Wahrscheinlichkeiten zu gut und der Gewinn zu optimistisch. Dafür nehmen wir cross_val_predict auf dem Train-Teil, das sind 108.394 Kunden, genau wie in 08_fehleranalyse.ipynb. Das dauert effektiv etwa eine halbe Minute.

In [ ]:
# Daten laden wie in 07_modelle.ipynb: Features und Split zusammen fügen, nur den Train-Teil nehmen
df = pd.read_csv(DATA_DIR / "fd_features.csv")
df_split = pd.read_csv(DATA_DIR / "fd_split.csv")
df = df.merge(df_split, on="client_id", how="left")
df_train = df[df["split"] == "train"]

X_train = df_train.drop(columns=["client_id", "creation_date", "split", "target"])
y_train = df_train["target"]

# Das beste Modell aus 07 laden. Mit clone() wir nehmen nur die Einstellungen, das sind die Hyperparameter,
# trainiert wird neu in jedem Fold. predict_proba gibt zwei Spalten zurück und `[:, 1]` ist die für "Betrug".
model = joblib.load(MODEL_DIR / "fd_best_model.joblib")
proba = cross_val_predict(clone(model), X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]

# Die Kunden nach Wahrscheinlichkeit sortieren: die verdächtigsten zuerst.
# Das ist die Liste, dass die Kontrolleure würden abarbeiten von oben nach unten.
df_liste = pd.DataFrame({"target": y_train.values, "proba": proba})
df_liste = df_liste.sort_values("proba", ascending=False).reset_index(drop=True)
df_liste.head()

Die ersten fünf Kunden auf der Liste haben alle eine Wahrscheinlichkeit über 0,97 und sie sind wirklich alle Betrüger, also target = 1. Das macht schon Sinn.

## Der Rechner
Wir gehen durch die Liste von oben nach unten und rechnen nach **jeder** Kontrolle:

* **Kosten** = Anzahl Kontrollen × Kosten pro Kontrolle
* **Ertrag** = gefundene Betrüger × Wert pro Betrug
* **Gewinn** = Ertrag − Kosten

Das Optimum ist dann einfach der Stelle wo der Gewinn ist am höchsten.

In [ ]:
def kontroll_rechner(df_liste, kosten, wert, max_kontrollen=None):
    # df_liste: sortierte Kundenliste, die verdächtigsten zuerst, mit der Spalte "target"
    # kosten:   Euro pro Kontrolle
    # wert:     Euro pro gefundenem Betrug
    # Gibt zurück die ganze Tabelle und die Zeile mit dem höchsten Gewinn, das ist das Optimum.
    tabelle = df_liste.copy()   # Kopie, damit die Original-Liste wird nicht verändert
    anzahl_kunden = len(tabelle)

    # Zeile 1 ist eine Kontrolle gemacht, Zeile 2 sind zwei Kontrollen gemacht, usw.
    tabelle["kontrollen"] = np.arange(1, anzahl_kunden + 1)
    tabelle["anteil_kontrolliert"] = tabelle["kontrollen"] / anzahl_kunden

    # cumsum zählt zusammen die Betrüger, die man hat gefunden bis zu diese Zeile
    tabelle["gefunden"] = tabelle["target"].cumsum()

    tabelle["kosten"] = tabelle["kontrollen"] * kosten
    tabelle["ertrag"] = tabelle["gefunden"] * wert
    tabelle["gewinn"] = tabelle["ertrag"] - tabelle["kosten"]

    # Kapazitätsgrenze von die Kontrolleure: dann zählt nur der obere Teil von der Liste
    if max_kontrollen is not None:
        tabelle = tabelle.head(max_kontrollen)

    # idxmax gibt der Index von die Zeile mit dem grössten Gewinn
    bester_index = tabelle["gewinn"].idxmax()
    optimum = tabelle.loc[bester_index]
    return tabelle, optimum


tabelle, optimum = kontroll_rechner(df_liste, KOSTEN_PRO_KONTROLLE, WERT_PRO_BETRUG, MAX_KONTROLLEN)

### 5 · Ergebnis

In [ ]:
alle_betrueger = df_liste["target"].sum()

# Break-even: ab welche Trefferquote kostet eine Kontrolle genau so viel wie sie bringt?
# Trefferquote * Wert = Kosten  ->  Trefferquote = Kosten / Wert
break_even = KOSTEN_PRO_KONTROLLE / WERT_PRO_BETRUG

# Die Zahlen vom Optimum einmal in eigene Variablen, dann die prints unten sind besser zu lesen
n_kontrollen = optimum["kontrollen"]
n_gefunden = optimum["gefunden"]
anteil_gefunden = n_gefunden / alle_betrueger        # Anteil an alle Betrüger
trefferquote = n_gefunden / n_kontrollen             # wie viele von die Kontrollen sind ein Treffer
gewinn_pro_kontrolle = optimum["gewinn"] / n_kontrollen

print(f"Eingaben: {KOSTEN_PRO_KONTROLLE} € pro Kontrolle, {WERT_PRO_BETRUG} € pro gefundenem Betrug")
print(f"Eine Kontrolle lohnt sich ab einer Trefferquote von {break_even:.1%}")
print()
if optimum["gewinn"] <= 0:
    # kann passieren wenn die Kosten sind sehr hoch und der Wert klein
    print("Ergebnis: Keine Kontrolle lohnt sich. Lieber gar nicht kontrollieren.")
else:
    print(f"Optimal: die verdächtigsten {optimum['anteil_kontrolliert']:.1%} der Kunden kontrollieren")
    print(f"  Kontrollen:          {n_kontrollen:,.0f}")
    print(f"  gefundene Betrüger:  {n_gefunden:,.0f} ({anteil_gefunden:.1%} aller Betrüger)")
    print(f"  Trefferquote:        {trefferquote:.1%}")
    print(f"  Kosten:              {optimum['kosten']:,.0f} €")
    print(f"  Ertrag:              {optimum['ertrag']:,.0f} €")
    print(f"  Gewinn:              {optimum['gewinn']:,.0f} €  ({gewinn_pro_kontrolle:.2f} € pro Kontrolle)")

**Was fällt auf?** Das gilt für die Standardwerte 25 € / 150 €:
* Eine Kontrolle lohnt sich ab einer Trefferquote von 16,7 %, das ist 25 ÷ 150.
* Optimal es ist, die **verdächtigsten 8,2 %** zu kontrollieren, das sind 8.920 Kunden. Fast jede dritte Kontrolle ist dann ein Treffer mit 30,1 % und man findet 44 % von alle Betrüger.
* Die Kosten sind 223.000 € und der Ertrag 402.300 €. Als Gewinn es bleiben rund **179.000 €**, also gut 20 € pro Kontrolle.
* Die Zahlen gelten für die 108.394 Kunden im Train-Teil. Für alle Kunden von STEG der Gewinn wäre entsprechend grösser.

## Gewinnkurve
Wie der Gewinn entwickelt sich, wenn man kontrolliert immer mehr Kunden? Zum Vergleich wir zeichnen auch ein, was passiert wenn man kontrolliert zufällig, also ohne Modell.

In [ ]:
quote_gesamt = df_liste["target"].mean()   # Betrugsquote über allen Kunden


def gewinn_bei_zufall(anzahl_kontrollen):
    # Wenn man kontrolliert zufällig, die Trefferquote ist immer die Betrugsquote von allen Kunden.
    # Eine Kontrolle bringt dann im Schnitt Quote * Wert und kostet KOSTEN_PRO_KONTROLLE.
    gewinn_pro_kontrolle_zufall = quote_gesamt * WERT_PRO_BETRUG - KOSTEN_PRO_KONTROLLE
    return anzahl_kontrollen * gewinn_pro_kontrolle_zufall


gewinn_zufall = gewinn_bei_zufall(tabelle["kontrollen"])

# x-Achse in Prozent und y-Achse in Tausend Euro, sonst die Zahlen an den Achsen sind zu lang
x_prozent = tabelle["anteil_kontrolliert"] * 100

plt.figure(figsize=(10, 5))
plt.plot(x_prozent, tabelle["gewinn"] / 1000, label="mit Modell")
plt.plot(x_prozent, gewinn_zufall / 1000, linestyle="--", label="zufällig kontrollieren")
plt.axhline(0, color="black", linewidth=1)   # Null-Linie: unter ihr ist Verlust

# das Optimum als ein rote Punkt
plt.scatter(optimum["anteil_kontrolliert"] * 100, optimum["gewinn"] / 1000, color="red", zorder=3,
            label=f"Optimum: {optimum['anteil_kontrolliert']:.1%} → {optimum['gewinn']:,.0f} €")

plt.title(f"Gewinn je nach Anzahl Kontrollen ({KOSTEN_PRO_KONTROLLE} € pro Kontrolle, {WERT_PRO_BETRUG} € pro Betrug)")
plt.xlabel("kontrollierte Kunden in %")
plt.ylabel("Gewinn in Tausend €")
plt.legend()
plt.show()

In [ ]:
# Im grossen Bild man sieht schlecht den Anfang, deshalb nochmal ein Zoom
# auf den interessanten Bereich, das sind die ersten 30 % der Kunden
bereich = tabelle[tabelle["anteil_kontrolliert"] <= 0.3]

plt.figure(figsize=(10, 4))
plt.plot(bereich["anteil_kontrolliert"] * 100, bereich["gewinn"] / 1000)
plt.axhline(0, color="black", linewidth=1)
plt.axvline(optimum["anteil_kontrolliert"] * 100, color="red", linestyle=":", label="Optimum")
plt.title("Zoom: die ersten 30 % der Kunden")
plt.xlabel("kontrollierte Kunden in %")
plt.ylabel("Gewinn in Tausend €")
plt.legend()
plt.show()

**Was fällt auf?**
* Am Anfang der Gewinn steigt steil, weil oben auf der Liste stehen die meisten Betrüger.
* Nach dem Optimum fällt er wider: ab da jede Kontrolle kostet mehr als sie bringt ein. Wer alle kontrolliert, macht sogar Verlust.
* **Zufällig kontrollieren** lohnt sich mit diesen Werten nie. Bei 5,6 % Treffern bringt eine Kontrolle im Schnitt nur ca. 8,40 € und kostet 25 €.
* Die Kurve ist um den Optimum herum ziemlich flach. Ob man 7 % oder 10 % kontrolliert macht kaum ein Unterschied.

## 7 · Warum genau dort aufhören?
Ganz oben auf die Liste ist ungefähr jeder zweite Kunde ein Betrüger. Je weiter man geht nach unten, desto seltener es wird. Wir zeigen hier die Trefferquote von den **jeweils nächsten 1.000 Kontrollen**.

Solange diese Trefferquote über der **Break-even-Linie** liegt, das ist Kosten ÷ Wert, bringt jede weitere Kontrolle mehr ein als sie kostet. Wenn sie darunter fällt macht jede weitere Kontrolle Verlust.

In [ ]:
# Die Trefferquote in ein "gleitenden Fenster" von 1.000 Kontrollen:
# `rolling(1000)` nimmt immer 1.000 Zeilen hintereinander und der `mean()` davon ist der Anteil Betrüger.
# center=True heißt, das Fenster liegt um die Zeile herum, 500 davor und 500 danach.
tabelle["trefferquote_naechste"] = tabelle["target"].rolling(1000, center=True).mean()

# nur die erste Hälfte von die Liste zeigen, danach nichts spannendes passiert mehr
bereich = tabelle[tabelle["anteil_kontrolliert"] <= 0.5]

plt.figure(figsize=(10, 4))
plt.plot(bereich["anteil_kontrolliert"] * 100, bereich["trefferquote_naechste"] * 100, label="Trefferquote der nächsten 1.000 Kontrollen")
plt.axhline(break_even * 100, color="red", linestyle="--", label=f"Break-even: {break_even:.1%}")
plt.axhline(quote_gesamt * 100, color="grey", linestyle=":", label=f"zufällig: {quote_gesamt:.1%}")
plt.title("Ab wann lohnt sich die nächste Kontrolle nicht mehr?")
plt.xlabel("kontrollierte Kunden in %")
plt.ylabel("Trefferquote in %")
plt.legend()
plt.show()

Das Optimum liegt da, wo die blaue Linie schneidet die rote Linie. Die blaue Linie wackelt ein bisschen, darum die Stelle im Bild ist nicht ganz scharf, aber es passt zu den 8,2 % von oben.

## Vergleich mit anderen Strategien

In [ ]:
def strategie_zeile(name, anzahl):
    # Ergebnis, wenn man kontrolliert die verdächtigsten "anzahl" Kunden.
    # iloc fängt an bei 0, deshalb anzahl - 1. Und int, weil zum Beispiel n * 0.05 ist eine Kommazahl
    t = tabelle.iloc[int(anzahl) - 1]
    return {"Strategie": name, "Kontrollen": int(t["kontrollen"]), "gefundene Betrüger": int(t["gefunden"]),
            "Trefferquote": round(t["gefunden"] / t["kontrollen"], 3), "Gewinn (€)": round(t["gewinn"])}


n = len(tabelle)
strategien = pd.DataFrame([
    strategie_zeile("Optimum laut Rechner", optimum["kontrollen"]),
    strategie_zeile("Top 5 %", n * 0.05),
    strategie_zeile("Top 10 %", n * 0.10),
    strategie_zeile("Top 20 %", n * 0.20),
    strategie_zeile("alle kontrollieren", n),
])

# Noch ein Zeile für "zufällig": gleich viele Kontrollen wie im Optimum aber ohne Modell.
# Die man kann nicht ablesen aus der Tabelle, deshalb wir rechnen sie mit der Betrugsquote aus.
k = optimum["kontrollen"]
strategien.loc[len(strategien)] = {
    "Strategie": "zufällig (gleich viele wie Optimum)", "Kontrollen": int(k),
    "gefundene Betrüger": round(k * quote_gesamt), "Trefferquote": round(quote_gesamt, 3),
    "Gewinn (€)": round(gewinn_bei_zufall(k)),
}

strategien

**Was fällt auf?**
* "Top 10 %", das ist unsere Schwelle aus 08_fehleranalyse.ipynb, ist fast so gut wie das genaue Optimum mit 175.000 € statt 179.000 €. Eine einfache Faustregel reicht also.
* "Top 5 %" hat die beste Trefferquote von 36,6 %, aber weniger Gewinn mit 162.000 €, weil man zu früh aufhört. Bei "Top 20 %" ist es umgekehrt, da man kontrolliert zu viele und es bleiben nur noch 81.900 €.
* **Mit Modell** +179.000 €, **zufällig** gleich viele Kontrollen: −148.000 €. Das Modell macht von ein Verlustgeschäft ein Gewinngeschäft.
* Alle zu kontrollieren macht 1,8 Mio. € Verlust.

## 9 · Was wäre wenn? Szenarien
Die Kosten und der Wert sind nur geschätzt. Wie die Empfehlung ändert sich, wenn sie sind anders? Wir lassen laufen den Rechner für verschiedene Kombinationen. Jedes Feld zeigt den **optimalen Anteil der Kunden**, den man soll kontrollieren.

In [ ]:
kosten_liste = [10, 25, 50, 100]             # Euro pro Kontrolle
wert_liste = [100, 150, 250, 500, 1000]      # Euro pro gefundene Betrug

# leere Tabelle: Zeilen sind Kosten, Spalten sind Wert
szenarien = pd.DataFrame(index=kosten_liste, columns=wert_liste, dtype=float)

for kosten in kosten_liste:
    for wert in wert_liste:
        # hier ohne Kapazitätsgrenze und die Tabelle selber wir brauchen nicht, deshalb der Unterstrich
        _, opt = kontroll_rechner(df_liste, kosten, wert)

        # wenn sogar im Optimum kommt kein Gewinn raus, die Empfehlung ist 0 % kontrollieren
        if opt["gewinn"] > 0:
            szenarien.loc[kosten, wert] = opt["anteil_kontrolliert"] * 100
        else:
            szenarien.loc[kosten, wert] = 0

plt.figure(figsize=(9, 4))
sns.heatmap(szenarien, annot=True, fmt=".0f", cmap="Blues", cbar_kws={"label": "% der Kunden kontrollieren"})
plt.title("Optimaler Anteil an Kontrollen in %")
plt.xlabel("Wert pro gefundenem Betrug (€)")
plt.ylabel("Kosten pro Kontrolle (€)")
plt.show()

# die genaue Zahlen noch als Tabelle, in der Heatmap sie sind gerundet auf ganze Prozent
szenarien.round(1)

**Was fällt auf?**
* Entscheidend ist nur das **Verhältnis** von Wert zu Kosten. Überall wo der Wert 10-mal so hoch ist wie die Kosten, zum Beispiel 10 € / 100 € und 50 € / 500 €, kommt das selbe heraus: 14,4 %.
* Ist eine Kontrolle teuer und der Betrug klein, also 100 € / 100 €, lohnt sich gar nichts.
* Ist eine Kontrolle billig und der Betrug groß, also 10 € / 1.000 €, dann es lohnt sich gut zwei Drittel von den Kunden zu kontrollieren, nämlich 69,9 %.
* Unser Standardfall 25 € / 150 € steht auch in die Tabelle, mit den 8,2 % von oben.

# Fazit
* Mit unseren Schätzungen, also 25 € pro Kontrolle und 150 € pro Betrug, sollte STEG **die verdächtigsten etwa 8–10 % der Kunden** kontrollieren.
* Damit findet sie fast **die Hälfte aller Betrüger** und fast jede dritte Kontrolle ist ein Treffer. Ohne Modell nur jede achtzehnte.
* Ohne Modell würden sich Kontrollen bei diesen Werten garnicht lohnen.

**Grenzen:**
* Kosten und Wert sind **geschätzt**. STEG kann in Abschnitt 2 die eigene Zahlen eintragen.
* Abschreckung ist nicht eingerechnet: wenn die Kontrollen sich sprechen herum, betrügen eventuell weniger Kunden. Dann lohnt sich Kontrollieren eher noch mehr.
* Ein Teil von den falschen Alarme sind vielleicht Betrüger, die bisher nur nie erwischt wurden. Die Rechnung ist also eher vorsichtig.

## 10 · Bonus: der Rechner zum Ausprobieren im Browser
Für die Präsentation wir haben noch ein kleines Tool gebaut. Das ist eine HTML Datei, die man kann öffnen in jedem Browser, auch ohne Python und ohne Jupyter. Man gibt ein die Kosten pro Kontrolle und die durchschnittliche Höhe von einem Betrug, und die Gewinnkurve ändert sich sofort, live. Die graue Linie zeigt immer die Ausgangswerte aus Abschnitt 2, so man sieht direkt den Unterschied.

Die Zelle unten schreibt die Datei kontroll_rechner.html in den Projektordner. Dann nur noch doppelklicken. Die Datei hat alles drin was sie braucht, es wird nichts aus dem Internet geladen.

In [ ]:
import json

# Für den Browser wir brauchen nicht die ganze Tabelle, das wäre viel zu groß.
# Es reicht zu wissen an welcher Stelle von der Liste ein Betrüger steht. Den Rest rechnet der Browser selber aus.
betrug_stellen = np.flatnonzero(df_liste["target"].to_numpy() == 1).tolist()

# Die Seite ist ein Text mit vier Platzhaltern, die ersetzen wir danach mit den echten Zahlen
VORLAGE = """<!doctype html>
<html lang="de">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Kontroll-Rechner</title>
<style>
:root { --petrol:#1f4e5f; --amber:#d98e04; --rot:#c0392b; --grau:#8a949b; --hell:#f1f4f5; --text:#1d2b32; }
* { box-sizing:border-box; }
body { margin:0; font-family:Arial, Helvetica, sans-serif; background:#fff; color:var(--text); }
.wrap { max-width:1280px; margin:0 auto; padding:20px 24px 28px; }
h1 { margin:0 0 4px; font-size:30px; color:var(--petrol); }
.sub { margin:0 0 16px; color:#55636b; font-size:16px; }
.eingaben { display:grid; grid-template-columns:1fr 1fr; gap:20px; margin-bottom:14px; }
.feld { background:var(--hell); border-radius:10px; padding:12px 16px; min-width:0; }
.feld label { display:block; font-weight:bold; font-size:17px; margin-bottom:6px; }
.zeile { display:flex; align-items:center; gap:12px; }
.zeile input[type=range] { flex:1; min-width:0; accent-color:var(--petrol); height:28px; }
.zeile input[type=number] { width:110px; min-width:0; font-size:26px; font-weight:bold; padding:4px 8px; border:2px solid var(--petrol); border-radius:8px; color:var(--petrol); text-align:right; }
.einheit { font-size:22px; font-weight:bold; color:var(--petrol); }
.leiste { display:flex; gap:10px; align-items:center; margin:4px 0 6px; flex-wrap:wrap; font-size:15px; }
.leiste .abstand { flex:1; }
button { font:inherit; padding:6px 14px; border-radius:8px; border:1px solid var(--petrol); background:#fff; color:var(--petrol); cursor:pointer; }
button.an { background:var(--petrol); color:#fff; }
label.haken { display:flex; align-items:center; gap:6px; cursor:pointer; }
svg { width:100%; height:auto; display:block; }
.kpis { display:grid; grid-template-columns:repeat(4, 1fr); gap:12px; margin-top:10px; }
.kpi { background:var(--hell); border-radius:10px; padding:10px 14px; }
.kpi .zahl { font-size:28px; font-weight:bold; color:var(--petrol); }
.kpi .text { font-size:14px; color:#55636b; margin-top:2px; }
.kpi.minus .zahl { color:var(--rot); }
#hinweis { display:none; background:#fdecea; border:1px solid var(--rot); color:var(--rot); border-radius:8px; padding:8px 12px; margin-top:10px; font-weight:bold; }
@media (max-width:800px) { .eingaben { grid-template-columns:1fr; } .kpis { grid-template-columns:1fr 1fr; } }
</style>
</head>
<body>
<div class="wrap">
  <h1>Kontroll-Rechner</h1>
  <p class="sub">Wie viele Kunden soll STEG kontrollieren? Werte ändern, die Kurve rechnet sofort neu.</p>

  <div class="eingaben">
    <div class="feld">
      <label for="k_zahl">Kosten pro Kontrolle</label>
      <div class="zeile">
        <input type="range" id="k_reg" min="1" max="200" step="1">
        <input type="number" id="k_zahl" min="0" step="1"><span class="einheit">€</span>
      </div>
    </div>
    <div class="feld">
      <label for="w_zahl">Durchschnittliche Betrugshöhe, Wert pro gefundenem Betrug</label>
      <div class="zeile">
        <input type="range" id="w_reg" min="10" max="1000" step="5">
        <input type="number" id="w_zahl" min="0" step="5"><span class="einheit">€</span>
      </div>
    </div>
  </div>

  <div class="leiste">
    <span>Ausschnitt:</span>
    <button type="button" data-zoom="30">erste 30 %</button>
    <button type="button" data-zoom="50">erste 50 %</button>
    <button type="button" data-zoom="100">alle Kunden</button>
    <span class="abstand"></span>
    <label class="haken"><input type="checkbox" id="geist" checked> Ausgangswerte als graue Linie zeigen</label>
    <button type="button" id="reset">Zurücksetzen</button>
  </div>

  <svg id="bild" viewBox="0 0 1000 430" role="img" aria-label="Gewinnkurve"></svg>
  <div id="hinweis">Mit diesen Werten lohnt sich keine einzige Kontrolle.</div>
  <div class="kpis" id="kpis"></div>
</div>

<script>
// Die Kundenliste ist nach Verdacht sortiert. BETRUG sind die Stellen in der Liste, an denen ein Betrüger steht.
const BETRUG = __BETRUG__;
const N = __ANZAHL__;
const K0 = __KOSTEN__;     // Ausgangswerte aus dem Notebook
const W0 = __WERT__;

// cum[k] = so viele Betrüger findet man, wenn man die ersten k Kunden kontrolliert
const cum = new Int32Array(N + 1);
let j = 0;
for (let k = 1; k <= N; k++) {
  if (j < BETRUG.length && BETRUG[j] === k - 1) j++;
  cum[k] = j;
}
const QUOTE = BETRUG.length / N;   // Betrugsquote über alle Kunden

const $ = id => document.getElementById(id);
const fmt = n => Math.round(n).toLocaleString("de-DE").replace("-", "−");
const eur = n => (n > 0 ? "+" : "") + fmt(n) + " €";
const proz = x => (x * 100).toLocaleString("de-DE", { minimumFractionDigits: 1, maximumFractionDigits: 1 }) + " %";

let zoom = 30;

// Optimum: die Kontrollmenge mit dem höchsten Gewinn
function optimum(kosten, wert) {
  let bestK = 0, bestG = 0;
  for (let k = 1; k <= N; k++) {
    const g = cum[k] * wert - k * kosten;
    if (g > bestG) { bestG = g; bestK = k; }
  }
  return { k: bestK, gewinn: bestG };
}

function schritt(spanne) {
  const roh = spanne / 6;
  const hoch = Math.pow(10, Math.floor(Math.log10(roh)));
  for (const m of [1, 2, 2.5, 5, 10]) if (m * hoch >= roh) return m * hoch;
  return 10 * hoch;
}

function karte(zahl, text, minus) {
  return '<div class="kpi' + (minus ? " minus" : "") + '"><div class="zahl">' + zahl + '</div><div class="text">' + text + "</div></div>";
}

function zeichne() {
  const kosten = Math.max(0, +$("k_zahl").value || 0);
  const wert = Math.max(0, +$("w_zahl").value || 0);
  const o = optimum(kosten, wert);
  const kmax = Math.round(N * zoom / 100);
  const sch = Math.max(1, Math.floor(kmax / 700));
  const ks = [];
  for (let k = 0; k <= kmax; k += sch) ks.push(k);
  if (ks[ks.length - 1] !== kmax) ks.push(kmax);

  const modell = ks.map(k => cum[k] * wert - k * kosten);
  const zufall = ks.map(k => k * (QUOTE * wert - kosten));
  const geistAn = $("geist").checked && (kosten !== K0 || wert !== W0);
  const geist = geistAn ? ks.map(k => cum[k] * W0 - k * K0) : [];

  // Beim Zoom darf die zufällige Linie unten aus dem Bild laufen, sonst sieht man das Optimum nicht mehr
  const fuerBereich = zoom === 100 ? [...modell, ...zufall, ...geist] : [...modell, ...geist];
  let ymin = Math.min(0, ...fuerBereich);
  let ymax = Math.max(0, ...fuerBereich);
  if (ymax - ymin < 1) ymax = ymin + 1;
  ymax += (ymax - ymin) * 0.12;   // etwas Platz für die Beschriftung oben
  const st = schritt(ymax - ymin);
  ymin = Math.floor(ymin / st) * st;
  ymax = Math.ceil(ymax / st) * st;

  const W = 1000, H = 430, L = 84, R = 20, T = 16, B = 52;
  const x = k => L + (k / kmax) * (W - L - R);
  const y = v => T + (1 - (v - ymin) / (ymax - ymin)) * (H - T - B);
  const linie = (werte, farbe, breite, dash) =>
    '<polyline fill="none" stroke="' + farbe + '" stroke-width="' + breite + '"' + (dash ? ' stroke-dasharray="8 6"' : "") +
    ' stroke-linejoin="round" points="' + ks.map((k, i) => x(k).toFixed(1) + "," + y(werte[i]).toFixed(1)).join(" ") + '"/>';

  let s = "";
  // Gitter und y-Achse in Tausend Euro
  for (let v = ymin; v <= ymax + st / 2; v += st) {
    s += '<line x1="' + L + '" x2="' + (W - R) + '" y1="' + y(v) + '" y2="' + y(v) + '" stroke="#e1e6e8"/>';
    s += '<text x="' + (L - 10) + '" y="' + (y(v) + 5) + '" text-anchor="end" font-size="15" fill="#55636b">' +
      (v / 1000).toLocaleString("de-DE", { maximumFractionDigits: 1 }).replace("-", "−") + "</text>";
  }
  // x-Achse in Prozent der Kunden
  const xs = zoom === 30 ? 5 : 10;
  for (let p = 0; p <= zoom; p += xs) {
    const k = N * p / 100;
    s += '<text x="' + x(k) + '" y="' + (H - B + 22) + '" text-anchor="middle" font-size="15" fill="#55636b">' + p + " %</text>";
  }
  s += '<text x="' + ((L + W - R) / 2) + '" y="' + (H - 8) + '" text-anchor="middle" font-size="16" fill="#1d2b32">kontrollierte Kunden in %</text>';
  s += '<text transform="translate(18 ' + ((T + H - B) / 2) + ') rotate(-90)" text-anchor="middle" font-size="16" fill="#1d2b32">Gewinn in Tausend €</text>';
  s += '<line x1="' + L + '" x2="' + (W - R) + '" y1="' + y(0) + '" y2="' + y(0) + '" stroke="#1d2b32" stroke-width="1.5"/>';

  s += '<clipPath id="flaeche"><rect x="' + L + '" y="' + T + '" width="' + (W - L - R) + '" height="' + (H - T - B) + '"/></clipPath><g clip-path="url(#flaeche)">';
  if (geistAn) s += linie(geist, "#b9c1c6", 3, false);
  s += linie(zufall, "#8a949b", 2.5, true);
  s += linie(modell, "#1f4e5f", 4, false);
  s += "</g>";

  // Optimum als Punkt mit Beschriftung
  if (o.k > 0 && o.k <= kmax) {
    const px = x(o.k), py = y(o.gewinn);
    s += '<line x1="' + px + '" x2="' + px + '" y1="' + py + '" y2="' + y(0) + '" stroke="#c0392b" stroke-dasharray="4 4"/>';
    s += '<circle cx="' + px + '" cy="' + py + '" r="8" fill="#c0392b" stroke="#fff" stroke-width="2"/>';
    const links = px > W * 0.62;
    s += '<text x="' + (links ? px - 14 : px + 14) + '" y="' + (py - 8) + '" text-anchor="' + (links ? "end" : "start") +
      '" font-size="17" font-weight="bold" fill="#c0392b">Optimum: ' + proz(o.k / N) + " → " + eur(o.gewinn) + "</text>";
  }

  // Legende
  let ly = T + 14;
  const leg = (farbe, dash, text) => {
    const t = '<line x1="' + (W - 330) + '" x2="' + (W - 296) + '" y1="' + (ly - 5) + '" y2="' + (ly - 5) + '" stroke="' + farbe +
      '" stroke-width="4"' + (dash ? ' stroke-dasharray="6 4"' : "") + '/><text x="' + (W - 288) + '" y="' + ly +
      '" font-size="15" fill="#1d2b32">' + text + "</text>";
    ly += 22;
    return t;
  };
  s += leg("#1f4e5f", false, "mit Modell");
  s += leg("#8a949b", true, "zufällig kontrollieren");
  if (geistAn) s += leg("#b9c1c6", false, "Ausgangswerte " + K0 + " € / " + W0 + " €");

  $("bild").innerHTML = s;

  // Kennzahlen
  const nutzen = o.k > 0;
  $("hinweis").style.display = nutzen ? "none" : "block";
  const gef = cum[o.k];
  const alle = BETRUG.length;
  const zufallGewinn = o.k * (QUOTE * wert - kosten);
  const be = wert > 0 ? kosten / wert : Infinity;
  $("kpis").innerHTML = nutzen
    ? karte(proz(o.k / N), "der Kunden kontrollieren, die verdächtigsten zuerst") +
      karte(fmt(o.k), "Kontrollen im Optimum") +
      karte(fmt(gef), "gefundene Betrüger, das sind " + proz(gef / alle) + " aller Betrüger") +
      karte(eur(o.gewinn), "Gewinn im Optimum") +
      karte(proz(gef / o.k), "Trefferquote im Optimum") +
      karte(isFinite(be) ? proz(be) : "–", "Break-even: so viele Treffer braucht jede Kontrolle mindestens") +
      karte(eur(zufallGewinn), "Gewinn bei gleich vielen zufälligen Kontrollen", zufallGewinn < 0) +
      karte(eur(o.gewinn / o.k), "Gewinn pro Kontrolle im Optimum")
    : karte("0 %", "kontrollieren") + karte("–", "Kontrollen") + karte("–", "gefundene Betrüger") + karte("0 €", "Gewinn");
}

// Regler und Zahlenfelder halten sich gegenseitig aktuell
function verbinde(reg, zahl) {
  $(reg).addEventListener("input", () => { $(zahl).value = $(reg).value; zeichne(); });
  $(zahl).addEventListener("input", () => { $(reg).value = $(zahl).value; zeichne(); });
}
verbinde("k_reg", "k_zahl");
verbinde("w_reg", "w_zahl");
$("geist").addEventListener("change", zeichne);

function setze(k, w) {
  $("k_zahl").value = k; $("k_reg").value = k;
  $("w_zahl").value = w; $("w_reg").value = w;
  zeichne();
}
$("reset").addEventListener("click", () => setze(K0, W0));

document.querySelectorAll("button[data-zoom]").forEach(b => {
  b.addEventListener("click", () => {
    zoom = +b.dataset.zoom;
    document.querySelectorAll("button[data-zoom]").forEach(c => c.classList.toggle("an", c === b));
    zeichne();
  });
});
document.querySelector('button[data-zoom="30"]').classList.add("an");

setze(K0, W0);
</script>
</body>
</html>
"""

html = VORLAGE
html = html.replace("__BETRUG__", json.dumps(betrug_stellen))
html = html.replace("__ANZAHL__", str(len(df_liste)))
html = html.replace("__KOSTEN__", str(KOSTEN_PRO_KONTROLLE))   # die Ausgangswerte von oben
html = html.replace("__WERT__", str(WERT_PRO_BETRUG))

ziel = Path("kontroll_rechner.html")
ziel.write_text(html, encoding="utf-8")
print(f"Gespeichert: {ziel.resolve()}")
print(f"{len(df_liste):,} Kunden, davon {len(betrug_stellen):,} Betrüger, Dateigröße {ziel.stat().st_size / 1000:,.0f} KB")